# Generative Models — Exercises

10 graded exercises covering VAEs, normalizing flows, GANs, diffusion models,
flow matching, score matching, and evaluation metrics.

| Format | Description |
|---|---|
| **Problem** | Markdown cell with task and equations |
| **Your Solution** | Scaffold code to complete |
| **Solution** | Reference implementation with PASS/FAIL checks |

### Difficulty Levels

| Level | Exercises | Focus |
|---|---|---|
| ★ | 1–3 | Core mechanics (ELBO, KL, reparam) |
| ★★ | 4–7 | Theory (flows, GANs, diffusion) |
| ★★★ | 8–10 | AI applications (FID, guidance, DSM) |

### Topic Map

| Topic | Exercises |
|---|---|
| VAE ELBO & KL | 1, 2 |
| Reparameterisation trick | 3 |
| Normalizing flows | 4 |
| GAN optimal discriminator | 5 |
| Diffusion forward process | 6 |
| DDIM sampling | 7 |
| FID computation | 8 |
| Classifier-free guidance | 9 |
| Denoising score matching | 10 |


In [ ]:
import numpy as np
from scipy.linalg import sqrtm
from scipy.special import logsumexp

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got     : {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def kl_gaussian(mu, log_var):
    """KL(N(mu, diag(exp(log_var))) || N(0,I)) per dimension then summed."""
    return 0.5 * np.sum(np.exp(log_var) + mu**2 - 1 - log_var)

def gaussian_log_prob(x, mu, log_var):
    """Log N(x; mu, diag(exp(log_var)))."""
    d = len(x)
    return -0.5 * (d*np.log(2*np.pi) + np.sum(log_var) + np.sum((x-mu)**2 / np.exp(log_var)))

print('Setup complete. NumPy:', np.__version__)

---

## Exercise 1 ★ — ELBO Decomposition

The VAE Evidence Lower Bound (ELBO) is:
$$\mathcal{L}_{\text{ELBO}} = \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})}[\log p_\theta(\mathbf{x}|\mathbf{z})] - D_{\text{KL}}(q_\phi(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}))$$

**Task**: Given the following encoder and decoder outputs, compute the ELBO.

- Encoder outputs: $\boldsymbol{\mu}_\phi = [0.5, -0.3]$, $\log\boldsymbol{\sigma}^2_\phi = [-0.2, 0.1]$
- Sampled latent: $\mathbf{z} = [0.4, -0.2]$ (use this fixed $\mathbf{z}$)
- Decoder output: reconstruction log-prob $= -1.8$ (given)

**(a)** Compute the KL term (closed form for diagonal Gaussian vs $\mathcal{N}(0,\mathbf{I})$).

**(b)** Compute the full ELBO using the given reconstruction term.

**(c)** What is the ELBO lower bound for $\log p(\mathbf{x})$?

In [ ]:
# Exercise 1: Your Solution
mu_phi = np.array([0.5, -0.3])
log_var_phi = np.array([-0.2, 0.1])
log_p_x_given_z = -1.8  # given

# (a) KL divergence
kl = None  # YOUR CODE HERE

# (b) ELBO
elbo = None  # YOUR CODE HERE

print(f'KL  = {kl}')
print(f'ELBO = {elbo}')

In [ ]:
# Exercise 1: Solution
mu_phi = np.array([0.5, -0.3])
log_var_phi = np.array([-0.2, 0.1])
log_p_x_given_z = -1.8

# (a) KL(N(mu, diag(sigma^2)) || N(0,I))
#   = 0.5 * sum(sigma^2 + mu^2 - 1 - log(sigma^2))
kl = 0.5 * np.sum(np.exp(log_var_phi) + mu_phi**2 - 1.0 - log_var_phi)

# (b) ELBO = E[log p(x|z)] - KL
elbo = log_p_x_given_z - kl

header('Exercise 1: ELBO Decomposition')
print(f'KL term  = {kl:.6f}')
print(f'Recon    = {log_p_x_given_z}')
print(f'ELBO     = {elbo:.6f}')

# Verify against formula
kl_expected = 0.5 * (np.exp(-0.2) + 0.25 - 1 + 0.2 + np.exp(0.1) + 0.09 - 1 - 0.1)
check_close('KL matches manual calculation', kl, kl_expected)
check_true('ELBO <= log p(x) (ELBO is a lower bound)', elbo <= 0.0)  # log p(x) <= 0 for bounded distributions
print('\nTakeaway: The KL term regularizes the encoder toward the prior; '
      'the ELBO balances reconstruction quality vs latent regularization.')

---

## Exercise 2 ★ — KL Divergence: Forward vs Reverse

Given two Gaussian distributions $p = \mathcal{N}(\mu_p, \sigma_p^2)$
and $q = \mathcal{N}(\mu_q, \sigma_q^2)$, the KL divergence is:
$$D_{\text{KL}}(p \| q) = \log\frac{\sigma_q}{\sigma_p} + \frac{\sigma_p^2 + (\mu_p - \mu_q)^2}{2\sigma_q^2} - \frac{1}{2}$$

Let $p = \mathcal{N}(2, 1)$ (data) and $q = \mathcal{N}(\mu, 1)$ (model).

**(a)** Compute $D_{\text{KL}}(p \| q)$ for $\mu \in \{0, 1, 2, 3, 4\}$.

**(b)** Show that $D_{\text{KL}}(p \| q) \geq 0$ with equality iff $p = q$.

**(c)** Compute $D_{\text{KL}}(q \| p)$ for the same $\mu$ values and compare.

In [ ]:
# Exercise 2: Your Solution
def kl_univariate_gaussian(mu_p, sig_p, mu_q, sig_q):
    # YOUR CODE HERE
    pass

mu_p, sig_p = 2.0, 1.0
sig_q = 1.0
mus = [0, 1, 2, 3, 4]

kl_forward = [kl_univariate_gaussian(mu_p, sig_p, mu, sig_q) for mu in mus]
kl_reverse = [kl_univariate_gaussian(mu, sig_q, mu_p, sig_p) for mu in mus]

print('mu   KL(p||q)  KL(q||p)')
for mu, f, r in zip(mus, kl_forward, kl_reverse):
    print(f'{mu}    {f}      {r}')

In [ ]:
# Exercise 2: Solution
def kl_univariate_gaussian(mu_p, sig_p, mu_q, sig_q):
    """KL(N(mu_p,sig_p^2) || N(mu_q,sig_q^2))."""
    return (np.log(sig_q/sig_p)
            + (sig_p**2 + (mu_p - mu_q)**2) / (2*sig_q**2)
            - 0.5)

mu_p, sig_p = 2.0, 1.0
sig_q = 1.0
mus = [0, 1, 2, 3, 4]

kl_fwd = [kl_univariate_gaussian(mu_p, sig_p, mu, sig_q) for mu in mus]
kl_rev = [kl_univariate_gaussian(mu, sig_q, mu_p, sig_p) for mu in mus]

header('Exercise 2: KL Forward vs Reverse')
print(f'  mu   KL(p||q)   KL(q||p)')
for mu, f, r in zip(mus, kl_fwd, kl_rev):
    print(f'  {mu}     {f:.6f}   {r:.6f}')

# (b) Non-negativity: KL >= 0, equality at mu=2
check_true('KL(p||q) >= 0 for all mu', all(kl >= 0 for kl in kl_fwd))
check_close('KL=0 when p=q (mu=2)', kl_fwd[2], 0.0)

# (c) Asymmetry: KL(p||q) != KL(q||p) in general
check_true('KL is asymmetric: KL(p||q) != KL(q||p) for mu!=mu_p',
           kl_fwd[0] != kl_rev[0])
print('\nTakeaway: Forward KL is mode-covering (MLE); '
      'reverse KL is mode-seeking (variational inference). '
      'Diffusion uses a KL chain; GANs approximate JS divergence.')

---

## Exercise 3 ★ — Reparameterisation Trick

The reparameterisation trick enables low-variance gradient estimates:
$$\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

The gradient $\nabla_\phi \mathbb{E}_{q_\phi(\mathbf{z})}[f(\mathbf{z})]$ can then be computed
by differentiating through the sampling operation.

**(a)** Implement the reparameterisation sampler.

**(b)** Estimate $\mathbb{E}_{q}[\|\mathbf{z}\|^2]$ via reparameterisation for
$\boldsymbol{\mu} = [1, 2]$, $\boldsymbol{\sigma} = [0.5, 1.0]$.

**(c)** Compare with the analytic value: $\mathbb{E}[\|\mathbf{z}\|^2] = \|\boldsymbol{\mu}\|^2 + \|\boldsymbol{\sigma}\|^2$.

In [ ]:
# Exercise 3: Your Solution
def reparam_sample(mu, log_var, n_samples=1):
    # YOUR CODE HERE: sample z using reparameterisation
    pass

mu = np.array([1.0, 2.0])
log_var = np.array([np.log(0.25), np.log(1.0)])  # log(sigma^2)

# (b) Estimate E[||z||^2]
samples = reparam_sample(mu, log_var, n_samples=10000)
E_norm_sq = None  # YOUR CODE HERE

print(f'E[||z||^2] (MC estimate) = {E_norm_sq}')

In [ ]:
# Exercise 3: Solution
def reparam_sample(mu, log_var, n_samples=1):
    """Sample z = mu + sigma * eps, eps ~ N(0,I)."""
    sigma = np.exp(0.5 * log_var)
    eps = np.random.randn(n_samples, len(mu))
    return mu + sigma * eps

mu = np.array([1.0, 2.0])
log_var = np.array([np.log(0.25), np.log(1.0)])  # sigma = [0.5, 1.0]
sigma = np.exp(0.5 * log_var)

np.random.seed(42)
samples = reparam_sample(mu, log_var, n_samples=50000)

# (b) MC estimate of E[||z||^2]
E_norm_sq_mc = np.mean(np.sum(samples**2, axis=1))

# (c) Analytic: E[||z||^2] = ||mu||^2 + ||sigma||^2
E_norm_sq_analytic = np.sum(mu**2) + np.sum(sigma**2)

header('Exercise 3: Reparameterisation Trick')
print(f'MC estimate:  {E_norm_sq_mc:.4f}')
print(f'Analytic:     {E_norm_sq_analytic:.4f}')
print(f'Difference:   {abs(E_norm_sq_mc - E_norm_sq_analytic):.4f}')

check_close('MC matches analytic E[||z||^2]', E_norm_sq_mc, E_norm_sq_analytic, tol=0.05)
check_close('Sample mean matches mu', samples.mean(axis=0), mu, tol=0.05)
check_close('Sample std matches sigma', samples.std(axis=0), sigma, tol=0.05)
print('\nTakeaway: Reparameterisation moves stochasticity out of the computation '
      'graph, enabling backpropagation through sampling — the key insight enabling VAE training.')

---

## Exercise 4 ★★ — Normalizing Flow: Change of Variables

For an invertible map $f: \mathbb{R}^d \to \mathbb{R}^d$, the change-of-variables formula gives:
$$\log p_X(x) = \log p_Z(f^{-1}(x)) + \log|\det J_{f^{-1}}(x)|$$

Consider the **affine coupling layer** (RealNVP-style):
$$y_{1:d/2} = x_{1:d/2}, \quad y_{d/2+1:d} = x_{d/2+1:d} \odot e^{s(x_{1:d/2})} + t(x_{1:d/2})$$

**(a)** Show that this is invertible and compute the log-determinant.

**(b)** Implement forward and inverse passes for $d=4$, with fixed scale and shift networks.

**(c)** Verify: $x \to y \to x$ recovers the original input.

In [ ]:
# Exercise 4: Your Solution
d = 4
d2 = d // 2

# Fixed scale/shift (in practice, neural networks)
np.random.seed(7)
W_s = np.random.randn(d2, d2) * 0.1
W_t = np.random.randn(d2, d2) * 0.1

def scale_net(x1):
    return np.tanh(x1 @ W_s)

def shift_net(x1):
    return x1 @ W_t

def coupling_forward(x):
    # YOUR CODE HERE
    # Returns: y, log_det
    pass

def coupling_inverse(y):
    # YOUR CODE HERE
    # Returns: x
    pass

x_test = np.array([1.0, -0.5, 0.3, 0.8])
y, log_det = coupling_forward(x_test)
x_rec = coupling_inverse(y)
print(f'x_original  = {x_test}')
print(f'y (forward) = {y}')
print(f'x_recovered = {x_rec}')
print(f'log_det     = {log_det}')

In [ ]:
# Exercise 4: Solution
d = 4
d2 = d // 2

np.random.seed(7)
W_s = np.random.randn(d2, d2) * 0.1
W_t = np.random.randn(d2, d2) * 0.1

def scale_net(x1):
    return np.tanh(x1 @ W_s)

def shift_net(x1):
    return x1 @ W_t

def coupling_forward(x):
    x1, x2 = x[:d2], x[d2:]
    s = scale_net(x1)
    t = shift_net(x1)
    y1 = x1
    y2 = x2 * np.exp(s) + t
    y = np.concatenate([y1, y2])
    log_det = np.sum(s)  # sum of log-diag of triangular Jacobian
    return y, log_det

def coupling_inverse(y):
    y1, y2 = y[:d2], y[d2:]
    s = scale_net(y1)
    t = shift_net(y1)
    x1 = y1
    x2 = (y2 - t) * np.exp(-s)
    return np.concatenate([x1, x2])

x_test = np.array([1.0, -0.5, 0.3, 0.8])
y, log_det = coupling_forward(x_test)
x_rec = coupling_inverse(y)

header('Exercise 4: Affine Coupling Layer')
print(f'x_original  = {x_test}')
print(f'y (forward) = {y}')
print(f'x_recovered = {x_rec}')
print(f'log_det     = {log_det:.6f}')

# Verify: first half unchanged, inverse recovers x
check_close('First half passthrough: y[:2] = x[:2]', y[:d2], x_test[:d2])
check_close('Inverse recovers x', x_rec, x_test)

# Numerical Jacobian verification
eps_jac = 1e-5
J_numerical = np.zeros((d, d))
for j in range(d):
    xp = x_test.copy(); xp[j] += eps_jac
    xm = x_test.copy(); xm[j] -= eps_jac
    yp, _ = coupling_forward(xp)
    ym, _ = coupling_forward(xm)
    J_numerical[:, j] = (yp - ym) / (2*eps_jac)

log_det_numerical = np.log(abs(np.linalg.det(J_numerical)))
check_close('log|det J| matches numerical Jacobian', log_det, log_det_numerical, tol=1e-4)
print('\nTakeaway: Triangular Jacobian structure makes log-det O(d) — '
      'the key efficiency trick enabling tractable normalizing flows.')

---

## Exercise 5 ★★ — GAN Optimal Discriminator

The optimal discriminator for the GAN minimax game is:
$$D^*(\mathbf{x}) = \frac{p_r(\mathbf{x})}{p_r(\mathbf{x}) + p_g(\mathbf{x})}$$

At this optimum, the generator loss equals:
$$\mathcal{L}_G = 2\,D_{\text{JS}}(p_r \| p_g) - \log 4$$

**(a)** Implement $D^*(x)$ for two 1-D Gaussians:
$p_r = \mathcal{N}(0, 1)$, $p_g = \mathcal{N}(1, 1)$.

**(b)** Verify: at $x = 0.5$ (equal densities), $D^*(0.5) = 0.5$.

**(c)** Compute the JS divergence numerically and verify $\mathcal{L}_G = 2\,D_{\text{JS}} - \log 4$.

In [ ]:
# Exercise 5: Your Solution
from scipy.stats import norm

def optimal_discriminator(x, mu_r=0.0, mu_g=1.0, sig=1.0):
    # YOUR CODE HERE
    pass

def js_divergence_numerical(mu_r=0.0, mu_g=1.0, sig=1.0, n=10000):
    # YOUR CODE HERE: approximate JS divergence via MC
    pass

x_test = 0.5
d_star = optimal_discriminator(x_test)
jsd = js_divergence_numerical()

print(f'D*(0.5)  = {d_star}')
print(f'JSD      = {jsd}')
print(f'L_G      = {2*jsd - np.log(4) if jsd else None}')

In [ ]:
# Exercise 5: Solution
from scipy.stats import norm
import numpy as np

def optimal_discriminator(x, mu_r=0.0, mu_g=1.0, sig=1.0):
    pr = norm.pdf(x, mu_r, sig)
    pg = norm.pdf(x, mu_g, sig)
    return pr / (pr + pg + 1e-300)

def js_divergence_numerical(mu_r=0.0, mu_g=1.0, sig=1.0, n=100000):
    """JS = (KL(p_r||m) + KL(p_g||m)) / 2, m = (p_r+p_g)/2"""
    xs = np.linspace(-5, 6, n)
    pr = norm.pdf(xs, mu_r, sig)
    pg = norm.pdf(xs, mu_g, sig)
    m  = 0.5*(pr + pg)
    eps = 1e-300
    kl_r = np.trapz(pr * np.log(pr / (m + eps) + eps), xs)
    kl_g = np.trapz(pg * np.log(pg / (m + eps) + eps), xs)
    return 0.5*(kl_r + kl_g)

header('Exercise 5: GAN Optimal Discriminator')
d_half = optimal_discriminator(0.5)  # equal densities at midpoint
d_left = optimal_discriminator(-3.0)  # dominated by real
d_right = optimal_discriminator(4.0)  # dominated by generated

print(f'D*(0.5)  = {d_half:.6f}  (expect 0.5)')
print(f'D*(-3.0) = {d_left:.6f}  (expect ~1.0 — real dominates)')
print(f'D*(4.0)  = {d_right:.6f}  (expect ~0.0 — generated dominates)')

jsd = js_divergence_numerical()
L_G = 2*jsd - np.log(4)
print(f'\nJS divergence (numerical) = {jsd:.6f}')
print(f'L_G = 2*JSD - log4         = {L_G:.6f}')

check_close('D*(0.5) = 0.5 at equal density point', d_half, 0.5, tol=0.001)
check_true('D*(x) close to 1 where real dominates', d_left > 0.95)
check_true('D*(x) close to 0 where gen dominates', d_right < 0.05)
check_true('JSD >= 0', jsd >= 0)
print('\nTakeaway: At GAN equilibrium, D* has a clean density-ratio form. '
      'When distributions overlap significantly, JSD is small and gradients vanish '
      '— this is the vanishing gradient problem solved by WGAN.')

---

## Exercise 6 ★★ — Diffusion Forward Marginal

The DDPM forward process has the closed-form marginal:
$$q(\mathbf{x}_t | \mathbf{x}_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t}\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I})$$

where $\bar{\alpha}_t = \prod_{s=1}^t (1-\beta_s)$ with linear schedule
$\beta_s \in [\beta_{\min}, \beta_{\max}]$.

**(a)** Implement the linear noise schedule and compute $\bar{\alpha}_t$.

**(b)** For $x_0 = 1.0$, compute the mean and variance of $q(x_t | x_0)$ at $t = 100, 500, 999$.

**(c)** Verify: as $t \to T$, $q(x_t | x_0)$ converges to $\mathcal{N}(0, \mathbf{I})$.

In [ ]:
# Exercise 6: Your Solution
T = 1000
beta_start, beta_end = 1e-4, 0.02

# (a) Build schedule
betas = None     # YOUR CODE HERE: linear schedule
alphas = None    # YOUR CODE HERE: 1 - betas
alpha_bar = None # YOUR CODE HERE: cumulative product

# (b) Compute mean and variance at key timesteps
x0 = 1.0
for t in [100, 500, 999]:
    mean_t = None  # YOUR CODE HERE
    var_t  = None  # YOUR CODE HERE
    print(f't={t}: mean={mean_t}, var={var_t}')

In [ ]:
# Exercise 6: Solution
T = 1000
beta_start, beta_end = 1e-4, 0.02

betas = np.linspace(beta_start, beta_end, T)
alphas = 1.0 - betas
alpha_bar = np.cumprod(alphas)

header('Exercise 6: Diffusion Forward Marginal')
x0 = 1.0
print(f'  t    sqrt(abar)    mean(x_t|x0)  var(x_t|x0)')
for t in [0, 100, 250, 500, 750, 999]:
    ab = alpha_bar[t]
    mean_t = np.sqrt(ab) * x0
    var_t  = 1.0 - ab
    print(f'  {t:<5}  {np.sqrt(ab):.6f}  {mean_t:.6f}      {var_t:.6f}')

# (c) Convergence to N(0,1) at T
mean_T = np.sqrt(alpha_bar[-1]) * x0
var_T  = 1.0 - alpha_bar[-1]

check_close('At T: mean -> 0', mean_T, 0.0, tol=0.05)
check_close('At T: var -> 1', var_T, 1.0, tol=0.05)

# SNR decreases monotonically
snr = alpha_bar / (1 - alpha_bar + 1e-10)
check_true('SNR monotonically decreasing', np.all(np.diff(snr) < 0))
print('\nTakeaway: The linear schedule drives q(x_T|x_0) to standard Gaussian. '
      'The SNR schedule affects sample quality; cosine schedules (Nichol & Dhariwal) '
      'are preferred for high-resolution images.')

---

## Exercise 7 ★★ — WGAN-GP: Gradient Penalty

The WGAN-GP objective adds a gradient penalty to enforce the 1-Lipschitz constraint:
$$\mathcal{L}_{\text{WGAN-GP}} = \mathbb{E}_{p_g}[D(\tilde{x})] - \mathbb{E}_{p_r}[D(x)]
+ \lambda\,\mathbb{E}_{\hat{x}}\left[(\|\nabla_{\hat{x}} D(\hat{x})\|_2 - 1)^2\right]$$

where $\hat{x} = \epsilon x + (1-\epsilon)\tilde{x}$ is sampled uniformly along straight lines.

**(a)** Implement the interpolation $\hat{x}$.

**(b)** For a linear discriminator $D(x) = w^T x$ (which is 1-Lipschitz iff $\|w\|=1$),
compute the gradient penalty.

**(c)** Verify: penalty = 0 when $\|\nabla_x D\| = 1$ everywhere.

In [ ]:
# Exercise 7: Your Solution
def interpolate(x_real, x_fake, eps):
    # YOUR CODE HERE
    pass

def gradient_penalty(D_fn, x_real, x_fake, lam=10.0, n=100):
    # YOUR CODE HERE
    # Numerically compute gradient penalty using finite differences
    pass

# Unit-norm linear discriminator: D(x) = w^T x, ||w||=1
d = 4
w = np.array([0.5, 0.5, 0.5, 0.5])  # ||w||=1
D_fn = lambda x: x @ w

np.random.seed(0)
x_real = np.random.randn(100, d)
x_fake = np.random.randn(100, d) + 1.0
gp = gradient_penalty(D_fn, x_real, x_fake)
print(f'Gradient penalty = {gp}')

In [ ]:
# Exercise 7: Solution
def interpolate(x_real, x_fake, eps):
    return eps * x_real + (1 - eps) * x_fake

def gradient_penalty(D_fn, x_real, x_fake, lam=10.0, n=100, delta=1e-5):
    """Numerically estimate gradient penalty."""
    N = min(len(x_real), len(x_fake), n)
    eps_samp = np.random.rand(N, 1)
    x_hat = interpolate(x_real[:N], x_fake[:N], eps_samp)
    d = x_hat.shape[1]
    # Finite-difference gradient estimate
    grads = np.zeros_like(x_hat)
    for j in range(d):
        xp = x_hat.copy(); xp[:, j] += delta
        xm = x_hat.copy(); xm[:, j] -= delta
        grads[:, j] = (np.array([D_fn(xp[i]) for i in range(N)])
                      - np.array([D_fn(xm[i]) for i in range(N)])) / (2*delta)
    grad_norms = np.linalg.norm(grads, axis=1)
    return lam * np.mean((grad_norms - 1.0)**2)

d = 4
w_unit = np.array([0.5, 0.5, 0.5, 0.5])  # ||w|| = 1
D_unit = lambda x: x @ w_unit

# Non-unit discriminator: ||w|| = 2 -> penalty should be large
w_big = np.array([1.0, 1.0, 1.0, 1.0])   # ||w|| = 2
D_big = lambda x: x @ w_big

np.random.seed(0)
x_real = np.random.randn(200, d)
x_fake = np.random.randn(200, d) + 1.0

gp_unit = gradient_penalty(D_unit, x_real, x_fake)
gp_big  = gradient_penalty(D_big, x_real, x_fake)

header('Exercise 7: WGAN-GP Gradient Penalty')
print(f'||w||=1: GP = {gp_unit:.4f}  (expect ~0)')
print(f'||w||=2: GP = {gp_big:.4f}   (expect large)')

check_true('Unit-norm discriminator has near-zero GP', gp_unit < 0.01)
check_true('Over-Lipschitz discriminator has large GP', gp_big > 1.0)
print('\nTakeaway: GP penalizes deviations from the 1-Lipschitz constraint. '
      'Spectral normalisation (SN-GAN) offers a per-layer Lipschitz constraint '
      'without the interpolation trick — now standard in modern GANs and transformers.')

---

## Exercise 8 ★★★ — Fréchet Inception Distance

FID measures the Fréchet distance between Gaussian-fitted feature distributions:
$$\text{FID} = \|\boldsymbol{\mu}_r - \boldsymbol{\mu}_g\|^2 + \text{tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r\Sigma_g)^{1/2})$$

**(a)** Implement FID using `scipy.linalg.sqrtm`.

**(b)** Verify: FID = 0 for identical distributions; FID increases with distribution shift.

**(c)** Estimate FID from samples: extract feature statistics from $N=2000$ random
feature vectors, comparing $p_r = \mathcal{N}(\mathbf{0}, \mathbf{I}_{64})$ vs
$p_g = \mathcal{N}(\boldsymbol{\mu}, \mathbf{I}_{64})$ for several values of $\|\boldsymbol{\mu}\|$.

In [ ]:
# Exercise 8: Your Solution
from scipy.linalg import sqrtm

def compute_fid(mu_r, sigma_r, mu_g, sigma_g):
    # YOUR CODE HERE
    pass

def fid_from_samples(samples_r, samples_g):
    # YOUR CODE HERE: compute FID from raw sample arrays (N x d)
    pass

d = 64
np.random.seed(42)
mu_r = np.zeros(d)
sigma_r = np.eye(d)

# Test at several shift magnitudes
for shift in [0.0, 0.5, 1.0, 2.0]:
    mu_g = np.ones(d) * shift / np.sqrt(d)  # unit vector * shift
    fid = compute_fid(mu_r, sigma_r, mu_g, sigma_r)
    print(f'shift={shift:.1f}: FID={fid}')

In [ ]:
# Exercise 8: Solution
from scipy.linalg import sqrtm

def compute_fid(mu_r, sigma_r, mu_g, sigma_g):
    diff = mu_r - mu_g
    covmean = sqrtm(sigma_r @ sigma_g)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return diff @ diff + np.trace(sigma_r + sigma_g - 2*covmean)

def fid_from_samples(samples_r, samples_g):
    mu_r = samples_r.mean(axis=0)
    mu_g = samples_g.mean(axis=0)
    sigma_r = np.cov(samples_r, rowvar=False)
    sigma_g = np.cov(samples_g, rowvar=False)
    return compute_fid(mu_r, sigma_r, mu_g, sigma_g)

d = 64
np.random.seed(42)

header('Exercise 8: FID Computation')

# (b) Analytical FID for identity covariances: FID = ||mu_r - mu_g||^2
sigma_I = np.eye(d)
fid_zero = compute_fid(np.zeros(d), sigma_I, np.zeros(d), sigma_I)
check_close('FID=0 for identical distributions', fid_zero, 0.0, tol=1e-6)

print(f'\n  shift  FID(analytic)  FID(from samples, N=2000)')
for shift in [0.0, 0.5, 1.0, 2.0]:
    mu_g = np.ones(d) * shift / np.sqrt(d)
    fid_analytic = compute_fid(np.zeros(d), sigma_I, mu_g, sigma_I)
    # Sample-based
    sr = np.random.randn(2000, d)
    sg = mu_g + np.random.randn(2000, d)
    fid_sample = fid_from_samples(sr, sg)
    print(f'  {shift:.1f}    {fid_analytic:.4f}         {fid_sample:.4f}')

check_true('FID increases with shift', (
    compute_fid(np.zeros(d), sigma_I, np.ones(d)*2/np.sqrt(d), sigma_I) >
    compute_fid(np.zeros(d), sigma_I, np.ones(d)*0.5/np.sqrt(d), sigma_I)
))
print('\nTakeaway: FID is the gold-standard metric for generative image quality. '
      'For identity covariances, FID = ||mu_r - mu_g||^2. '
      'Real evaluation uses Inception-v3 features on 50K samples.')

---

## Exercise 9 ★★★ — Classifier-Free Guidance

Classifier-free guidance (CFG) modifies the score estimate:
$$\hat{\boldsymbol{\epsilon}} = \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t, \emptyset)
+ w \cdot \left[\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t, c)
- \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t, \emptyset)\right]$$

**(a)** Show that $w=0$ recovers unconditional and $w=1$ recovers conditional sampling.

**(b)** For a toy 1-D diffusion model, implement CFG sampling with different $w$ values
and show that higher $w$ concentrates samples near the conditional mode.

**(c)** Compute the effective score $\nabla_{x_t}\log p(x_t|c)^{1+w}$ interpretation.

In [ ]:
# Exercise 9: Your Solution
def cfg_eps(eps_cond, eps_uncond, w):
    # YOUR CODE HERE
    pass

# Verify: w=0 gives uncond, w=1 gives cond
eps_c = np.array([0.8, -0.3])
eps_u = np.array([0.1, 0.2])

for w in [0.0, 1.0, 3.0, 7.5]:
    result = cfg_eps(eps_c, eps_u, w)
    print(f'w={w}: eps_hat = {result}')

In [ ]:
# Exercise 9: Solution
def cfg_eps(eps_cond, eps_uncond, w):
    return eps_uncond + w * (eps_cond - eps_uncond)

eps_c = np.array([0.8, -0.3])
eps_u = np.array([0.1, 0.2])

header('Exercise 9: Classifier-Free Guidance')
for w in [0.0, 1.0, 3.0, 7.5]:
    result = cfg_eps(eps_c, eps_u, w)
    print(f'  w={w:.1f}: eps_hat = {result}')

# (a) Verify special cases
check_close('w=0 recovers unconditional', cfg_eps(eps_c, eps_u, 0.0), eps_u)
check_close('w=1 recovers conditional',   cfg_eps(eps_c, eps_u, 1.0), eps_c)

# (b) Toy simulation: N(mu_c, 1) conditional, N(0, 1) unconditional
# CFG with high w should push samples toward mu_c
np.random.seed(42)
mu_c_true = 5.0
n_steps = 200

def simulate_cfg(w, n_samples=500):
    x = np.random.randn(n_samples)
    for t in range(n_steps, 0, -1):
        snr = t / n_steps  # proxy for alpha_bar
        # Oracle scores
        eps_unc = -x  # score of N(0,1)
        eps_con = -(x - mu_c_true * snr)  # score of N(mu_c * snr, 1)
        eps_hat = cfg_eps(eps_con, eps_unc, w)
        x = x - 0.01 * eps_hat + np.sqrt(0.005) * np.random.randn(n_samples)
    return x

print(f'\n  w      mean(samples)  (target: mu_c={mu_c_true})')
for w in [0.0, 1.0, 3.0, 7.5]:
    samps = simulate_cfg(w)
    print(f'  {w:.1f}    {samps.mean():.3f}')

print('\nTakeaway: CFG amplifies the conditional score at the cost of diversity. '
      'w=7.5 is the Stable Diffusion default. Too high w causes artifacts '
      '(clipping issue) — addressed by dynamic thresholding in Imagen.')

---

## Exercise 10 ★★★ — Denoising Score Matching

DSM (Vincent 2011) trains a score network via:
$$\mathcal{J}_{\text{DSM}}(\theta) = \mathbb{E}_{x \sim p_{\text{data}}, \tilde{x} \sim q_\sigma(\cdot|x)}
\left[\|s_\theta(\tilde{x}) - \nabla_{\tilde{x}}\log q_\sigma(\tilde{x}|x)\|^2\right]$$

where $q_\sigma(\tilde{x}|x) = \mathcal{N}(x, \sigma^2 I)$, so the target score is
$\nabla_{\tilde{x}}\log q_\sigma(\tilde{x}|x) = (x - \tilde{x})/\sigma^2$.

**(a)** Implement the DSM loss for a toy linear score network $s_\theta(x) = Wx + b$.

**(b)** Fit the score network to a mixture of two Gaussians via gradient descent.

**(c)** Verify the learned score matches the analytic score at several test points.

In [ ]:
# Exercise 10: Your Solution
def dsm_loss(W, b, data_batch, sigma=0.3):
    # YOUR CODE HERE
    # Add Gaussian noise to data, compute DSM loss
    # score_theta(x_tilde) = W * x_tilde + b  (scalar case)
    pass

# Generate bimodal data
np.random.seed(42)
N = 2000
data = np.concatenate([np.random.randn(N//2) - 2, np.random.randn(N//2) + 2])

# Initialize parameters
W, b = -0.1, 0.0

# One step of gradient descent (implement full training loop)
loss = dsm_loss(W, b, data[:100])
print(f'Initial DSM loss: {loss}')

In [ ]:
# Exercise 10: Solution
def score_network(x, W, b):
    return W * x + b

def dsm_loss(W, b, data_batch, sigma=0.3):
    n = len(data_batch)
    eps = np.random.randn(n) * sigma
    x_tilde = data_batch + eps
    # DSM target: (x - x_tilde) / sigma^2
    target = -eps / sigma**2  # = (x - x_tilde) / sigma^2
    predicted = score_network(x_tilde, W, b)
    return np.mean((predicted - target)**2)

def dsm_grad(W, b, data_batch, sigma=0.3):
    """Numerical gradient of DSM loss."""
    delta = 1e-5
    L = lambda w, bb: dsm_loss(w, bb, data_batch, sigma)
    gW = (L(W+delta, b) - L(W-delta, b)) / (2*delta)
    gb = (L(W, b+delta) - L(W, b-delta)) / (2*delta)
    return gW, gb

np.random.seed(42)
N = 2000
data = np.concatenate([np.random.randn(N//2) - 2, np.random.randn(N//2) + 2])

# Train linear score network s(x) = W*x + b
W, b = 0.0, 0.0
lr = 0.01
batch_size = 256
losses = []

for step in range(500):
    idx = np.random.choice(N, batch_size)
    gW, gb = dsm_grad(W, b, data[idx])
    W -= lr * gW
    b -= lr * gb
    if step % 100 == 0:
        losses.append(dsm_loss(W, b, data, sigma=0.3))

header('Exercise 10: Denoising Score Matching')
print(f'Trained W={W:.4f}, b={b:.4f}')
print(f'Loss progression: {[f"{l:.4f}" for l in losses]}')

# Compare with analytic score at test points
# Analytic: s(x) = p'(x)/p(x) for 0.5*N(-2,1) + 0.5*N(2,1)
def analytic_score(x):
    p1 = 0.5*np.exp(-0.5*(x+2)**2)/np.sqrt(2*np.pi)
    p2 = 0.5*np.exp(-0.5*(x-2)**2)/np.sqrt(2*np.pi)
    return (p1*(-(x+2)) + p2*(-(x-2))) / (p1 + p2)

# Linear model can't fit bimodal score perfectly, but should trend correctly
test_pts = np.array([-3.0, -2.0, 0.0, 2.0, 3.0])
print(f'\n  x    s_learned  s_analytic  sign_match')
for x in test_pts:
    s_learn = score_network(x, W, b)
    s_true  = analytic_score(x)
    match = 'YES' if np.sign(s_learn) == np.sign(s_true) else 'NO'
    print(f'  {x:>5.1f}  {s_learn:>9.4f}  {s_true:>10.4f}  {match}')

# Sign should match on most points (linear approximation)
signs_match = sum(np.sign(score_network(x, W, b)) == np.sign(analytic_score(x))
                  for x in test_pts)
check_true('Linear score matches analytic sign on majority of points', signs_match >= 3)
print('\nTakeaway: DSM is equivalent to ISM (Hyvärinen 2005) and is the '
      'foundation of diffusion model training. The DDPM objective is DSM at noise level sigma_t '
      '— reparameterised as eps-prediction for variance reduction.')

---

## What to Review After Finishing

- [ ] **ELBO** — can you derive it from $\log p(x) \geq \mathcal{L}_{\text{ELBO}}$ using Jensen's inequality?
- [ ] **Reparameterisation** — understand why the score-function gradient has high variance
- [ ] **Normalizing flows** — verify that coupling layers are O(d) in log-det computation
- [ ] **GAN equilibrium** — derive the optimal $D^*$ from first principles
- [ ] **Diffusion** — reconstruct the DDPM simplified loss from the full ELBO
- [ ] **Score matching** — understand how DSM = ISM (up to a constant)
- [ ] **FID** — know when FID fails (mode collapse can give low FID)
- [ ] **CFG** — derive the guidance formula from Bayes' rule and the classifier gradient

## References

1. Kingma & Welling (2013). *Auto-Encoding Variational Bayes*. arXiv:1312.6114
2. Dinh et al. (2016). *Density estimation using Real-valued Non-Volume Preserving*. arXiv:1605.08803
3. Goodfellow et al. (2014). *Generative Adversarial Networks*. arXiv:1406.2661
4. Ho et al. (2020). *Denoising Diffusion Probabilistic Models*. arXiv:2006.11239
5. Song et al. (2021). *Score-Based Generative Modeling through SDEs*. arXiv:2011.13456
6. Ho & Salimans (2021). *Classifier-Free Diffusion Guidance*. arXiv:2207.12598
7. Lipman et al. (2022). *Flow Matching for Generative Modeling*. arXiv:2210.02747
8. Heusel et al. (2017). *GANs Trained by a Two Time-Scale Update Rule*. arXiv:1706.08500
9. Arjovsky et al. (2017). *Wasserstein GAN*. arXiv:1701.07875
10. Vincent (2011). *A Connection Between Score Matching and Denoising Autoencoders*. NC
